In [1]:
import sys
sys.path.append('..')

from models.compression.vq_vae import VQVAE
from dataset import ImageCompressionDataModule

In [2]:
device = "cuda"

model = VQVAE(
    h_dim=256,
    n_embeddings=1024,
    embedding_dim=64,
    beta=0.25,
    lr=1e-3,
    #
    #
    #
    n_heads=8,
    n_layers=8,
    patch_size=2,
)

model = model.to(device)

AssertionError: learnable codebook not compatible with EMA update

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.tuner import Tuner

dataset = ImageCompressionDataModule(
    data_dir="../../data/PetImages",
    batch_size=2,
)

dataset.setup()

model = VQVAE(
    h_dim=56,
    res_h_dim=256,
    n_res_layers=3,
    n_embeddings=512 * 2,
    embedding_dim=16,
    beta=0.25,
    lr=1e-3,
    num_downsamples=3,
    initial_channels=128,
).to(device)

trainer = pl.Trainer(
    max_epochs=1,
    accelerator=device,
    devices=1,
    logger=False,
    precision="bf16-mixed"
)

tuner = Tuner(trainer)
lr_finder = tuner.lr_find(
    model, 
    datamodule=dataset,
    # min_lr=1e-6,
    # max_lr=1e-1,
    num_training=20,
)

fig = lr_finder.plot(suggest=True)
fig.show()

print(f"Suggested LR: {lr_finder.suggestion()}")